# Submission: LightGBM baseline with the building

**lgbm_building** alone, to score the baseline on the leaderboard next to the blend in
[submission.ipynb](../../ensemble/notebooks/submission.ipynb). Same model as the blend's LightGBM half: LightGBM on the 33 features plus
`building_id` as a categorical, untuned (500 trees, learning rate 0.1, 63 leaves). On the 4 seasonal folds it scored
1.217, against 1.187 for mlp_4out_emb and 1.178 for the blend ([ensemble.ipynb](../../ensemble/notebooks/ensemble.ipynb)).

- **Training rows:** the same 2M rows as [submission.ipynb](../../ensemble/notebooks/submission.ipynb): sampled from all of 2016 with anomalies
  dropped, same seed.
- **No torch here,** so LightGBM can use every core (next to torch it crashes unless limited to one thread).

Output: `data/processed/submission_lgbm_building.csv`, with site 0 electricity converted back to kBTU, the unit Kaggle
scores.


In [1]:
import sys

import lightgbm as lgb
import numpy as np
import pandas as pd

sys.path.insert(0, "../..")  # the ashrae package lives in the homework folder
from ashrae.config import PROCESSED
from ashrae.data import read_processed
from ashrae.features import build_features
from ashrae.metrics import rmsle
from ashrae.submission import build_submission, check_submission, to_readings
from ashrae.tracking import log_run

N_TRAIN = 2_000_000  # the same rows as the blend in submission.ipynb
CHUNK_SIZE = 1_000_000  # test rows predicted at once
LGBM_PARAMS = {"n_estimators": 500, "learning_rate": 0.1, "num_leaves": 63}
SUBMISSION_PATH = PROCESSED / "submission_lgbm_building.csv"

2026/09/27 22:59:35 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at /Users/lupaskodmitro/FuckIT/Machine Learning/Deep Learning Course/.venv/lib/python3.14/site-packages/mlflow/assistant/skills/instrumenting-with-mlflow-tracing/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


## Training rows

Sampled from all of 2016 without anomalies ([EDA §9](../../eda/notebooks/EDA.ipynb#broken)), as in
[submission.ipynb](../../ensemble/notebooks/submission.ipynb). The `year_built` median comes from these rows and is reused for the test set.


In [2]:
df = read_processed()
BUILDING = pd.CategoricalDtype(
    range(int(df.building_id.max()) + 1)
)  # all ids, so train and test share categories
train_df = df[~df.anomaly].sample(N_TRAIN, random_state=0)
del df

year_median = train_df.year_built.median()
X_train = build_features(train_df, year_median)
y_train = np.log1p(train_df.meter_reading.to_numpy(dtype="float32"))
print(f"{len(train_df):,} training rows, {X_train.shape[1]} features + building_id")


def with_building(X, rows):
    """`X` plus `building_id` as a categorical, which LightGBM splits on directly."""
    building = rows.building_id.astype(int).astype(BUILDING)
    return X.assign(building_id=building.to_numpy())

2,000,000 training rows, 33 features + building_id


## Model


In [ ]:
X_train_lgbm = with_building(X_train, train_df)
lgbm = lgb.LGBMRegressor(**LGBM_PARAMS, random_state=0, verbose=-1).fit(
    X_train_lgbm, y_train
)
train_rmsle = rmsle(y_train, lgbm.predict(X_train_lgbm))

log_run(
    "ashrae-submission",
    f"lgbm_building | {N_TRAIN:,} rows | submission",
    params={**LGBM_PARAMS, "n_train": N_TRAIN},
    metrics={"train_rmsle": train_rmsle},
    tags={"model": "lgbm_building", "notebook": "submission_baseline"},
)

print(f"lgbm_building: training RMSLE {train_rmsle:.3f}")

lgbm_building: training RMSLE 0.833


## Test predictions

The 41.7M test rows in chunks of 1M. `log1p` predictions are clipped at 0 and turned back into readings with `expm1`.


In [4]:
test_df = read_processed("test_merged.parquet")


def predict_log(rows):
    """Clipped log1p predictions for `rows`, built with the same feature columns as training."""
    X = build_features(rows, year_median).reindex(columns=X_train.columns, fill_value=0)
    return np.clip(lgbm.predict(with_building(X, rows)), 0, None)


starts = range(0, len(test_df), CHUNK_SIZE)
lgbm_log = np.concatenate(
    [predict_log(test_df.iloc[start : start + CHUNK_SIZE]) for start in starts]
)
print(f"mean log1p prediction: {lgbm_log.mean():.3f}")

mean log1p prediction: 4.234


In [5]:
reading = to_readings(lgbm_log, test_df)
submission = build_submission(test_df, reading)
check_submission(submission)

submission.to_csv(SUBMISSION_PATH, index=False, float_format="%.4f")
print(f"saved {SUBMISSION_PATH}: {len(submission):,} rows")
submission.meter_reading.describe().round(2).to_frame()

saved /Users/lupaskodmitro/FuckIT/Machine Learning/Deep Learning Course/Homeworks/Homework 1/data/processed/submission_lgbm_building.csv: 41,697,600 rows


,meter_reading
count,41697600.00
mean,376.83
std,2710.44
min,0.00
25%,22.76
50%,76.16
75%,229.89
max,1734915.79


## Next

Upload `submission_lgbm_building.csv` to Kaggle and compare with the blend (private 1.302, public 1.086). If the
leaderboard ranks them like the folds did (1.217 vs 1.178), the fold validation at least ranks models correctly.
